# Build & publish the INDEPENDENT TEST set  (temp / maintainer notebook)

Self-contained (only this notebook ships to Colab). Collects a houseplant-disease
**test set** from the same sources the training notebook uses (+ Openverse CC),
de-dups, balances, and **auto-uploads / versions** the private Kaggle dataset
`maxayyt5/houseplant-disease-testset` — which `train_notebook.ipynb` is meant to
grade against (re-add the validation cell there, or use the optional last cell here).

**Provenance tiers** (tagged per image in `manifest.csv -> disjoint_from_train`):
- `openverse` (CC) = **disjoint** from training — the trustworthy part.
- `doctorp` / other Kaggle slugs = **TRAINING sources -> LEAKY**; included for class
  coverage. **pHash-dedup them against the training split before trusting as a gate.**

Token is read from a **secret** (`KAGGLE_API_TOKEN`), never hardcoded.


In [ ]:
# === §1 Config ===========================================================
CLASS_NAMES = ['gray_mold_botrytis','healthy','leaf_scorch_underwatering','leaf_spot',
               'not_a_plant','nutrient_deficiency','pests','powdery_mildew','viral_disease']
TEST_SLUG   = 'maxayyt5/houseplant-disease-testset'   # create-or-version target

PER_CLASS_CAP   = 40        # max kept per class (balance + bound size)
IMG_MAX         = 768       # downscale long side (optimized validation/transfer)
PHASH_THRESHOLD = 6         # Hamming <= -> near-duplicate (drop)
SEED            = 0

# Which sources to draw from. openverse=disjoint(CC); the rest are TRAINING slugs
# (leaky) reused from the training notebook for class coverage.
USE_OPENVERSE   = True
USE_DOCTORP     = True      # alexanderuzhinskiy/the-doctorp-project-dataset (leaky)
DISJOINT_ONLY   = False     # True -> Openverse only (clean but sparse)
# Openverse keyword search is UNJUDGED -> reliable only for healthy/not_a_plant.
# Its disease results are noisy; enable only if you will manually cull them.
OPENVERSE_DISEASE = False

DO_UPLOAD       = True      # set False for a dry run (build only, no Kaggle write)
OUT_DIR         = 'testset_build'

In [ ]:
# === §2 Kaggle auth from a SECRET (new-format API token) ==================
# Order: Colab secret KAGGLE_API_TOKEN -> env -> ~/.kaggle/access_token.
import os
def _load_token():
    try:
        from google.colab import userdata          # Colab secret (recommended)
        t = userdata.get('KAGGLE_API_TOKEN')
        if t: return t.strip()
    except Exception:
        pass
    if os.environ.get('KAGGLE_API_TOKEN'):
        return os.environ['KAGGLE_API_TOKEN'].strip()
    p = os.path.expanduser('~/.kaggle/access_token')
    if os.path.exists(p):
        return open(p).read().strip()
    return None

_TOKEN = _load_token()
assert _TOKEN, ("No Kaggle token. In Colab add a secret named KAGGLE_API_TOKEN, "
                "or set the env var / write ~/.kaggle/access_token.")
import kagglehub
from kagglehub.config import set_kaggle_api_token
set_kaggle_api_token(_TOKEN)                        # bearer auth for download + upload
print('Kaggle token loaded (bearer).')

In [ ]:
# === §3 Helpers: identity, pHash dedup, threaded download, downscale ======
import io, hashlib, urllib.request, urllib.parse, json, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from PIL import Image
UA = {'User-Agent': 'plantvision-testset/1.0 (research)'}

def _http(url, timeout=25):
    return urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=timeout).read()

_DCT = None
def _dct_basis(n=32):
    k = np.arange(n); b = np.cos(np.pi*(2*k[:,None]+1)*k[None,:]/(2*n)); b[0]/=np.sqrt(2); return b*np.sqrt(2/n)
def phash(img):
    global _DCT
    if _DCT is None: _DCT = _dct_basis(32)
    g = np.asarray(img.convert('L').resize((32,32), Image.BILINEAR), dtype=np.float64)
    low = (_DCT @ g @ _DCT.T)[:8,:8]
    bits = (low > np.median(low.flatten()[1:])).flatten()
    h = 0
    for x in bits: h = (h<<1)|int(x)
    return h
def hamming(a,b): return bin(a^b).count('1')

def fetch_image(url):
    """Download -> RGB PIL (downscaled). None on failure/too-small."""
    try:
        im = Image.open(io.BytesIO(_http(url))).convert('RGB')
        if min(im.size) < 96: return None
        im.thumbnail((IMG_MAX, IMG_MAX))
        return im
    except Exception:
        return None

def fetch_many(urls, workers=12):
    out = []
    with ThreadPoolExecutor(max_workers=workers) as ex:
        for u, im in zip(urls, ex.map(fetch_image, urls)):
            if im is not None: out.append((u, im))
    return out

In [ ]:
# === §4 Sources (reused from the training notebook + Openverse) ===========
# 4a. Openverse (Creative-Commons) -> DISJOINT from training.
OPENVERSE_Q = {
 'gray_mold_botrytis':['botrytis gray mold leaf','grey mould plant','botrytis blight flower'],
 'healthy':['healthy houseplant leaf','potted plant green leaves'],
 'leaf_scorch_underwatering':['leaf scorch','leaf tip burn brown','underwatered plant dry leaves'],
 'leaf_spot':['leaf spot disease','fungal leaf spot plant','bacterial leaf spot'],
 'not_a_plant':['living room sofa','kitchen interior','cat pet','wooden table'],
 'nutrient_deficiency':['leaf chlorosis yellow','iron deficiency plant leaf','interveinal chlorosis'],
 'pests':['mealybug plant','spider mites leaf damage','scale insect plant','aphids on leaf'],
 'powdery_mildew':['powdery mildew leaf','powdery mildew plant'],
 'viral_disease':['mosaic virus leaf','plant virus leaf mottling','leaf curl virus'],
}
def openverse_urls(q, n=20, page=1):
    u = f"https://api.openverse.org/v1/images/?q={urllib.parse.quote(q)}&page_size={n}&page={page}&mature=false"
    for _ in range(3):
        try: return [r['url'] for r in json.loads(_http(u)).get('results',[])]
        except Exception: time.sleep(2)
    return []

# 4b. DoctorP (Kaggle TRAINING slug) -> folder->class map copied from train_notebook §2.
DOCTORP_SLUG = 'alexanderuzhinskiy/the-doctorp-project-dataset'
DOCTORP_MAP = {
 'grey mold':'gray_mold_botrytis','botrytis cinerea':'gray_mold_botrytis',
 'scale':'pests','mealybug':'pests',
 'marginal leaf necrosis':'leaf_scorch_underwatering','burn':'leaf_scorch_underwatering','leaves scorch':'leaf_scorch_underwatering',
 'mosaic virus':'viral_disease','powdery mildew':'powdery_mildew','nutrient deficiency':'nutrient_deficiency',
 'healthy':'healthy','cherry leaf spot':'leaf_spot','bacterial spot':'leaf_spot','leaf spot':'leaf_spot','anthracnose':'leaf_spot',
}
def doctorp_paths():
    """{class: [local image paths]} from the DoctorP Kaggle dataset (curated labels)."""
    root = kagglehub.dataset_download(DOCTORP_SLUG)
    out = {}
    for dp,_,files in os.walk(root):
        cls = DOCTORP_MAP.get(os.path.basename(dp).strip().lower())
        if not cls: continue
        out.setdefault(cls, []).extend(sorted(os.path.join(dp,f) for f in files
                                              if f.lower().endswith(('.jpg','.jpeg','.png'))))
    return out

In [ ]:
# === §5 Build the balanced, de-duped test set ============================
import shutil, csv, random
rng = random.Random(SEED)
shutil.rmtree(OUT_DIR, ignore_errors=True)
rows, counts, seen_ph = [], {c:0 for c in CLASS_NAMES}, []   # global pHash dedup

def _accept(cls, im, source, disjoint, source_url, note):
    if counts[cls] >= PER_CLASS_CAP: return False
    ph = phash(im)
    if any(hamming(ph,h) <= PHASH_THRESHOLD for h in seen_ph): return False
    seen_ph.append(ph)
    d = f'{OUT_DIR}/{cls}'; os.makedirs(d, exist_ok=True)
    fn = f'{cls}_{counts[cls]:02d}.jpg'; im.save(f'{d}/{fn}', quality=88)
    counts[cls] += 1
    rows.append({'file':f'{cls}/{fn}','class':cls,'verified_by':'agent_or_curated',
                 'source':source,'disjoint_from_train':str(disjoint).lower(),
                 'source_url':source_url,'note':note})
    return True

# Openverse first (disjoint) so it wins the dedup ties.
if USE_OPENVERSE:
    for cls in CLASS_NAMES:
        if cls not in ('healthy','not_a_plant') and not OPENVERSE_DISEASE:
            continue   # unjudged Openverse disease results are too noisy
        for q in OPENVERSE_Q.get(cls, []):
            if counts[cls] >= PER_CLASS_CAP: break
            for url, im in fetch_many(openverse_urls(q, n=20)):
                if counts[cls] >= PER_CLASS_CAP: break
                _accept(cls, im, 'openverse', True, url, 'CC, best-effort disjoint')
    print('after openverse:', {c:counts[c] for c in CLASS_NAMES})

# DoctorP (leaky training source) to fill the classes Openverse can't supply.
if USE_DOCTORP and not DISJOINT_ONLY:
    dp = doctorp_paths()
    for cls in CLASS_NAMES:
        paths = dp.get(cls, []); rng.shuffle(paths)
        for p in paths:
            if counts[cls] >= PER_CLASS_CAP: break
            try: im = Image.open(p).convert('RGB'); im.thumbnail((IMG_MAX,IMG_MAX))
            except Exception: continue
            _accept(cls, im, 'doctorp', False, 'https://www.kaggle.com/datasets/'+DOCTORP_SLUG,
                    'TRAINING SOURCE - pHash-dedup vs train REQUIRED before gate use')
    print('after doctorp:', {c:counts[c] for c in CLASS_NAMES})

with open(f'{OUT_DIR}/manifest.csv','w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=['file','class','verified_by','source','disjoint_from_train','source_url','note'])
    w.writeheader(); w.writerows(rows)
total = sum(counts.values()); dj = sum(1 for r in rows if r['disjoint_from_train']=='true')
open(f'{OUT_DIR}/README.md','w').write(
 f"# Houseplant disease - INDEPENDENT TEST set\n\n"
 f"{total} images / {sum(1 for c in counts if counts[c])} classes  "
 f"({dj} disjoint=openverse, {total-dj} leaky=training-source).\n\n"
 f"Per-class: {json.dumps(counts)}\n\n"
 f"`disjoint_from_train=false` rows are from TRAINING sources (leaky) - pHash-dedup "
 f"against the train split, or replace with pipeline-collected disjoint data, before "
 f"treating this as a gate.\n")
print('BUILT', total, 'images |', dj, 'disjoint |', total-dj, 'leaky')
print(json.dumps(counts, indent=1))

In [ ]:
# === §6 Auto-create / update the private Kaggle dataset ===================
# Same handle -> a new VERSION; first run -> creates it (private by default).
if DO_UPLOAD:
    kagglehub.dataset_upload(
        TEST_SLUG, OUT_DIR,
        version_notes=f'auto: {sum(counts.values())} imgs / '
                      f'{sum(1 for c in counts if counts[c])} classes '
                      f'({sum(1 for r in rows if r["disjoint_from_train"]=="true")} disjoint)')
    print('uploaded ->', f'https://kaggle.com/datasets/{TEST_SLUG}')
else:
    print('DO_UPLOAD=False -> built locally only, not uploaded.')

In [ ]:
# === §7 (optional) Validate an exported bundle against this test set ======
# Relocated from the training notebook. Point BUNDLE_DIR at a models/ dir that
# has classifier(_int8).onnx + classes.json; grades it on the freshly built set.
BUNDLE_DIR = 'models'        # e.g. an extracted plantvision_*.tar
RUN_VALIDATION = False
if RUN_VALIDATION:
    import onnxruntime as ort
    from sklearn.metrics import f1_score
    meta = json.load(open(f'{BUNDLE_DIR}/classes.json'))
    cls_names = meta['class_names']; size = (meta.get('training') or {}).get('img_size',224)
    ratio = (meta.get('training') or {}).get('eval_resize_ratio',1.05)
    mean = np.array([0.485,0.456,0.406],np.float32); std = np.array([0.229,0.224,0.225],np.float32)
    onx = next((f'{BUNDLE_DIR}/{n}' for n in ('classifier_int8.onnx','classifier.onnx')
                if os.path.exists(f'{BUNDLE_DIR}/{n}')), None)
    sess = ort.InferenceSession(onx, providers=['CPUExecutionProvider']); inp = sess.get_inputs()[0].name
    def prep(im):
        s = max(size, int(size*ratio)); w,h = im.size; sc = s/min(w,h)
        im = im.resize((max(size,round(w*sc)),max(size,round(h*sc))), Image.BILINEAR)
        l=(im.size[0]-size)//2; t=(im.size[1]-size)//2; im=im.crop((l,t,l+size,t+size))
        a=(np.asarray(im,np.float32)/255.0-mean)/std; return a.transpose(2,0,1)[None]
    yt, yp = [], []
    for ci,cls in enumerate(cls_names):
        d=f'{OUT_DIR}/{cls}'
        if not os.path.isdir(d): continue
        for fn in os.listdir(d):
            lo=sess.run(None,{inp:prep(Image.open(f'{d}/{fn}').convert('RGB'))})[0]
            yp.append(int(lo.argmax())); yt.append(ci)
    yt,yp=np.array(yt),np.array(yp)
    di=[i for i,n in enumerate(cls_names) if n not in ('healthy','not_a_plant')]
    f1=f1_score(yt,yp,labels=di,average='macro',zero_division=0)
    rng2=np.random.default_rng(SEED)
    bs=[f1_score(yt[s],yp[s],labels=di,average='macro',zero_division=0)
        for s in (rng2.integers(0,len(yt),len(yt)) for _ in range(2000))]
    print(f'INDEPENDENT TEST disease_macro_f1={f1:.3f} '
          f'95% CI [{np.percentile(bs,2.5):.3f}, {np.percentile(bs,97.5):.3f}] n={len(yt)}')